# Mosquito Supermodel: Predict Demo

Run YOLOv11 `predict` on the bundled demo assets and produce detections plus a merged `results.csv`. Use this as a quick-start to validate your install or as a reference for wiring the API into your own workflows.

**Requirements**
- Install the package from the repo root: `pip install -e .`
- Make sure that you are on the right interpreter
- Make sure YOLito weights are at `weights/last.pt` 
- The demo media lives in `src/demo` (`blood_feeding.png`, `dead.jpeg`, `flying.MOV`).

In [4]:
import sys
sys.path.append('../')
from pathlib import Path
import yaml

from mosquito_supermodel import build_runtime_config, run_task


def find_repo_root(start: Path) -> Path:
    # Walk upward until we find the repo root that contains configs/.
    for candidate in [start, *start.parents]:
        if (candidate / "configs").exists():
            return candidate
    return start


repo_root = find_repo_root(Path.cwd())
demo_dir = repo_root  / "demo"
weights_path = repo_root / "weights" / "last.pt"
output_dir = repo_root / "output" / "demo_predict"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"repo_root={repo_root}")
print(f"demo_dir={demo_dir}")
print(f"weights_path={weights_path}")
print(f"output_dir={output_dir}")



repo_root=/home/bohbot/Evyatar/git/Mosquito_Supermodel
demo_dir=/home/bohbot/Evyatar/git/Mosquito_Supermodel/demo
weights_path=/home/bohbot/Evyatar/git/Mosquito_Supermodel/weights/last.pt
output_dir=/home/bohbot/Evyatar/git/Mosquito_Supermodel/output/demo_predict


In [5]:
# Choose which demo asset to run. Use demo_dir to process the whole folder.
image_or_video = demo_dir / "blood_feeding.png"

config = {
    "images_dir": str(image_or_video),
    "model": {
        "conf_threshold": 0.25,
        "iou_threshold": 0.5,
        "task": "predict",  # options: predict | track | slice
        "vid_stride": 1,
        "weights": str(weights_path),
    },
    "output_dir": str(output_dir),
    "sahi": {
        "overlap_ratio": 0.1,
        "slice_size_h": 640,
        "slice_size_w": 640,
        "track": True,
    },
    "change_analyze_conf": False,
    "save_animations": True,
}

config_path = output_dir / "infer_demo.yaml"
config_path.write_text(yaml.safe_dump(config))
config_path



PosixPath('/home/bohbot/Evyatar/git/Mosquito_Supermodel/output/demo_predict/infer_demo.yaml')

In [6]:
runtime_config = build_runtime_config("infer", config_path)
run_task(runtime_config)



Exported config to /home/bohbot/Evyatar/git/Mosquito_Supermodel/output/demo_predict/infer_config.yaml
No frames in video: /home/bohbot/Evyatar/git/Mosquito_Supermodel/demo/blood_feeding.png

image 1/1 /home/bohbot/Evyatar/git/Mosquito_Supermodel/demo/blood_feeding.png: 640x480 74 1s, 35.4ms
Speed: 5.7ms preprocess, 35.4ms inference, 53.1ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /home/bohbot/Evyatar/git/Mosquito_Supermodel/output/demo_predict/predict
Results saved to: /home/bohbot/Evyatar/git/Mosquito_Supermodel/output/demo_predict
Results saved to /home/bohbot/Evyatar/git/Mosquito_Supermodel/output/demo_predict/predict/results.csv
No CSVs to merge.
Removed empty folder: /home/bohbot/Evyatar/git/Mosquito_Supermodel/output/demo_predict/csvs
Removed empty folder: /home/bohbot/Evyatar/git/Mosquito_Supermodel/output/demo_predict/predict
Removed empty folder: /home/bohbot/Evyatar/git/Mosquito_Supermodel/output/demo_predict/videos


In [7]:
import pandas as pd

results_csv = Path(output_dir) / "results.csv"
display(pd.read_csv(results_csv).head())

frames_dir = Path(output_dir) / "frames"
predictions = sorted(frames_dir.glob("*.jpg"))
print(f"Annotated frames: {len(predictions)}")
if predictions:
    from IPython.display import Image, display as show
    show(Image(filename=predictions[0]))



,image_idx,box_idx,x,y,w,h,confidence,label,track_id,image_name,img_h,img_w
0,0,0,2156.5652,4505.55570,310.26330,355.85790,0.884671,0.0,NaN,blood_feeding,8534,6401
1,0,1,1511.5354,3502.87840,312.08887,313.61816,0.877069,0.0,NaN,blood_feeding,8534,6401
2,0,2,1268.3923,741.32745,438.70776,410.63098,0.873356,0.0,NaN,blood_feeding,8534,6401
3,0,3,1511.5994,4820.36500,321.95300,376.11963,0.865416,0.0,NaN,blood_feeding,8534,6401
4,0,4,2083.8730,4026.75220,268.60315,357.22803,0.863204,0.0,NaN,blood_feeding,8534,6401


Annotated frames: 3
